# 11 Dashboard: data cleaning
Numbers for the dashboard's **Data cleaning** page. Replays the code cells of `02_data_cleaning.ipynb` one at a time (same idea as section 1 of `99_dups.ipynb`) and records after each step:
- how many rows it removed
- how many missing values it filled, per column
- which columns it dropped
- which rows of a 20k raw sample it dropped (for the "what got removed" scatter)

`02` is not changed, and its save cell (`to_csv`) is skipped, so `vehicles_clean.csv` is never rewritten. Output: small CSVs in `data/dashboard/`.

In [1]:
import json
import numpy as np
import pandas as pd
from carpricepredictor.shared import CLEAN_CSV, DASHBOARD_DIR

## Step names
Each `02` heading gets the name shown on the dashboard and its rule in one line. A heading that isn't listed here (e.g. a new step added to `02` later) shows up as-is.

In [2]:
# 02 heading -> (name on the dashboard, the rule in one line)
STEPS = {
    "1 loading data and dependencies": ("raw listings", "Craigslist used-car listings across the US, posted April–May 2021"),
    "Droping useless cols": ("unused columns", "id, image_url, region_url, posting_date and county dropped: ids, links, one month of dates, an empty column"),
    "Miising price = drop": ("$0 price", "listings with a $0 price (no price given)"),
    "Droping classic cars": ("before 1980", "built before 1980 or no year: classics follow different pricing"),
    "droping region ins in between state": ("region column", "the ~400 craigslist regions dropped, state is kept instead"),
    "size = drop": ("size column", "dropped: it overlaps with model"),
    "Price": ("fake price", "under $500, under $1,000 for a 2010+ car, over $200k, or a 2021+ car under $5k (a financing price)"),
    "condition filling": ("condition", "missing → 'unknown'"),
    "paint_color filling + joining fancy ones": ("paint colour", "missing → 'missing', rare colours → 'not ordinary'"),
    "brands": ("manufacturer", "missing → first word of the model if it's a known brand (typos like 'chevorlet' fixed), else 'unknown'"),
    "Year col": ("missing year", "no year (already removed by the 1980 step)"),
    "Model": ("model names", "raw model text → clean base model with Claude; rare names → 'other', missing → 'unknown'"),
    "Odometer": ("bad odometer", "missing, over 500,000 miles, or under 100 miles on a pre-2020 car"),
    "Fuel just drop na": ("missing fuel", "no fuel type"),
    "title_status": ("missing title", "no title status"),
    "Cylinders": ("unknown cylinders", "electric → 'other', hybrid → 4 cylinders, else the model's most common; still unknown → dropped"),
    "Transmission its a very smal na (1.4k)": ("missing transmission", "no transmission"),
    "Type": ("type", "missing → the model's most common type, else 'unknown'"),
    "drive (same method as type)": ("drive", "missing → the model's drive when 80%+ of its listings agree, else 'unknown'"),
    "finnally droping dupes": ("duplicates", "same values in every column except state and the raw listing fields: the same car posted again"),
    "Iteration 2: listing fields": ("listing fields", "url, VIN, description and lat / long turned into is_dealer, has_vin, desc_len and keyword flags"),
}
PLACEHOLDERS = {"unknown", "missing", "other"}

## Replay 02 step by step
Before each `02` code cell: the row count and which values are missing. After it:
- **removed** = rows before − rows after
- **filled** = values that were missing before and aren't after, counted only on rows the step kept (`to_placeholder` = how many of them became `unknown` / `missing` / `other`, and which one)
- dropped columns, and `dropped_at` for the sample rows that just disappeared

`02` only removes rows, never adds them, so every row left after a step was also there before it.

In [3]:
cells = json.load(open("02_data_cleaning.ipynb"))["cells"]

steps, fills, column_dropped_at = [], [], {}
heading, df = None, None
for cell in cells:
    src = "".join(cell["source"])
    if cell["cell_type"] == "markdown":
        if src.strip():
            heading = src.split("\n")[0].strip("# ").strip()
        continue
    if "to_csv" in src:  # 02's save cell
        continue
    label, rule = STEPS.get(heading, (heading, ""))

    if df is None:  # the imports cell, then the cell that reads vehicles.csv
        exec(src)
        if df is not None:
            raw_missing = df.isna().mean()
            sample = df[["price", "year", "odometer"]].sample(20_000, random_state=42).assign(dropped_at="kept")
            steps.append({"step": label, "rule": rule, "rows_before": len(df), "rows_after": len(df)})
        continue

    rows_before, missing_before = len(df), df.isna()
    exec(src)

    cols = missing_before.columns.intersection(df.columns)
    now_filled = missing_before.loc[df.index, cols] & df[cols].notna()
    for col in [c for c in cols if now_filled[c].any()]:
        values = df.loc[now_filled[col], col]
        placeholders = values[values.isin(PLACEHOLDERS)]
        fills.append({"column": col, "step": label, "rule": rule, "filled": len(values),
                      "to_placeholder": len(placeholders),
                      "placeholder": placeholders.mode().iloc[0] if len(placeholders) else ""})
    for col in missing_before.columns.difference(df.columns):
        column_dropped_at[col] = label
    gone = ~sample.index.isin(df.index) & (sample["dropped_at"] == "kept")
    sample.loc[gone, "dropped_at"] = label
    steps.append({"step": label, "rule": rule, "rows_before": rows_before, "rows_after": len(df)})

steps = pd.DataFrame(steps)
steps["removed"] = steps["rows_before"] - steps["rows_after"]
fills = pd.DataFrame(fills)
steps[["step", "rows_before", "rows_after", "removed"]]

,step,rows_before,rows_after,removed
0,raw listings,426880,426880,0
1,unused columns,426880,426880,0
2,$0 price,426880,393985,32895
3,before 1980,393985,384591,9394
4,region column,384591,384591,0
5,size column,384591,384591,0
6,fake price,384591,372712,11879
7,condition,372712,372712,0
8,paint colour,372712,372712,0
9,manufacturer,372712,372712,0


In [4]:
fills[["column", "step", "filled", "to_placeholder", "placeholder"]]

,column,step,filled,to_placeholder,placeholder
0,condition,condition,140276,140276,unknown
1,paint_color,paint colour,106119,106119,missing
2,manufacturer,manufacturer,12480,5111,unknown
3,model,model names,3695,3695,unknown
4,cylinders,unknown cylinders,147709,1603,other
5,type,type,75537,1010,other
6,drive,drive,105697,42695,unknown


## Checks
The replay must end with exactly as many rows as `vehicles_clean.csv`.

In [5]:
clean_rows = len(pd.read_csv(CLEAN_CSV, usecols=["price"]))
raw_rows = steps["rows_before"].iloc[0]
assert steps["rows_after"].iloc[-1] == clean_rows, (steps["rows_after"].iloc[-1], clean_rows)
print(f"{raw_rows:,} raw listings -> {clean_rows:,} clean ({clean_rows / raw_rows:.1%} kept)")
print("rows dropped by the sample:", sample["dropped_at"].value_counts().to_dict())

426,880 raw listings -> 205,385 clean (48.1% kept)
rows dropped by the sample: {'kept': 9636, 'duplicates': 7095, '$0 price': 1497, 'fake price': 584, 'before 1980': 430, 'bad odometer': 285, 'missing title': 282, 'missing fuel': 106, 'missing transmission': 53, 'unknown cylinders': 32}


## Missing values: raw vs clean
Share of missing values per raw column, before and after the cleaning, and how they were handled:
- `dropped_at`: the step that dropped the column (empty = kept)
- `became`: for url / VIN / description, the listing fields they were turned into before the raw text was dropped
- `how`: turned into listing fields / column dropped / filled / rows dropped (the rows with it missing were removed) / left missing (the model's imputer fills it) / never missing

In [6]:
# the raw listing columns 02 turns into model features (listing_fields) before dropping the raw text
BECAME = {"url": "is_dealer", "VIN": "has_vin", "description": "desc_len + 11 keyword flags"}

missing = pd.DataFrame({
    "raw_missing": raw_missing,
    "clean_missing": df.isna().mean().reindex(raw_missing.index),
    "dropped_at": pd.Series(column_dropped_at).reindex(raw_missing.index),
}).rename_axis("column").reset_index()
missing = missing.join(fills.groupby("column")[["filled", "to_placeholder"]].sum(), on="column")
missing["became"] = missing["column"].map(BECAME)
missing["how"] = np.select(
    [missing["became"].notna(), missing["dropped_at"].notna(), missing["filled"] > 0,
     missing["clean_missing"] > 0, missing["raw_missing"] > 0],
    ["turned into listing fields", "column dropped", "filled", "left missing", "rows dropped"],
    default="never missing",
)
missing.round(3)

,column,raw_missing,clean_missing,dropped_at,filled,to_placeholder,became,how
0,id,0.000,NaN,unused columns,NaN,NaN,NaN,column dropped
1,url,0.000,NaN,listing fields,NaN,NaN,is_dealer,turned into listing fields
2,region,0.000,NaN,region column,NaN,NaN,NaN,column dropped
3,region_url,0.000,NaN,unused columns,NaN,NaN,NaN,column dropped
4,price,0.000,0.000,NaN,NaN,NaN,NaN,never missing
5,year,0.003,0.000,NaN,NaN,NaN,NaN,rows dropped
6,manufacturer,0.041,0.000,NaN,12480.0,5111.0,NaN,filled
7,model,0.012,0.000,NaN,3695.0,3695.0,NaN,filled
8,condition,0.408,0.000,NaN,140276.0,140276.0,NaN,filled
9,cylinders,0.416,0.000,NaN,147709.0,1603.0,NaN,filled


## Model names
From `01_data_cleaning_model_names.ipynb`: every raw (manufacturer, model) name seen 3+ times and the clean model Claude mapped it to, with how many listings use it.
- **spellings**: the 15 clean models with the most raw spellings (`unknown` left out)
- **examples**: the 5 with the most spellings, each with its 6 most-listed raw spellings

In [7]:
names = pd.read_csv("data/model_map.csv")
counts = pd.read_csv("data/model_map_input.csv")
assert names[["manufacturer", "model_raw"]].equals(counts[["manufacturer", "model_raw"]])
names["listings"] = counts["count"]

unknown = names["model_clean"] == "unknown"
print(f"{len(names):,} raw names -> {names['model_clean'].nunique():,} clean models, "
      f"{names.loc[unknown, 'listings'].sum() / names['listings'].sum():.1%} of listings -> unknown")

by_model = names[~unknown].groupby(["manufacturer", "model_clean"])
spellings = (pd.DataFrame({"spellings": by_model.size(), "listings": by_model["listings"].sum()})
             .sort_values("spellings", ascending=False).head(15).reset_index())

top5 = spellings.head(5)[["manufacturer", "model_clean"]].assign(rank=range(5))
examples = (names.merge(top5).sort_values(["rank", "listings"], ascending=[True, False])
            .groupby("rank").head(6).drop(columns="rank"))
spellings

9,763 raw names -> 935 clean models, 0.9% of listings -> unknown


,manufacturer,model_clean,spellings,listings
0,ford,f-150,228,13760
1,ram,1500,157,8087
2,ford,f-250,149,6280
3,ford,f-350,146,4656
4,bmw,3 series,128,4116
5,toyota,tacoma,98,5105
6,chevrolet,silverado 1500,89,8145
7,jeep,wrangler,85,7017
8,gmc,sierra 1500,85,5077
9,ram,2500,85,3446


In [8]:
examples

,manufacturer,model_raw,model_clean,listings
128,ford,f-150,f-150,6917
130,ford,f150,f-150,1039
139,ford,f-150 xlt,f-150,399
141,ford,f150 supercrew cab xlt,f-150,343
143,ford,f150 super cab xl pickup 4d,f-150,289
144,ford,f150 supercrew cab,f-150,265
651,ram,1500,1500,3545
652,ram,1500 crew cab big horn,1500,635
653,ram,1500 classic regular cab,1500,383
654,ram,pickup 1500,1500,252


## Save

In [9]:
DASHBOARD_DIR.mkdir(exist_ok=True)
out = {
    "cleaning_steps.csv": steps,
    "cleaning_fills.csv": fills,
    "cleaning_sample.csv": sample,
    "missing_values.csv": missing,
    "model_names.csv": names,
    "model_spellings.csv": spellings,
    "model_examples.csv": examples,
}
for name, table in out.items():
    table.to_csv(DASHBOARD_DIR / name, index=False)
    print(f"{name:22} {len(table):>6,} rows")

cleaning_steps.csv         21 rows
cleaning_fills.csv          7 rows
cleaning_sample.csv    20,000 rows
missing_values.csv         26 rows
model_names.csv         9,763 rows
model_spellings.csv        15 rows
model_examples.csv         30 rows
